# Combined Drought Propagation Chain Figure
**Left panel:** SSI time-series strips (single column, earliest upstream → origin)  
**Right panel:** Spatial map of the Ebro Basin with station categories  

Run all cells — the last cell will prompt you interactively for station and chain.

In [ ]:
# ── Cell 1: Imports ───────────────────────────────────────────────────────────
import os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from datetime import timedelta
from scipy.interpolate import make_interp_spline

mpl.rcParams.update({
    'font.family'      : 'serif',
    'font.serif'       : ['Times New Roman', 'DejaVu Serif', 'serif'],
    'font.size'        : 8,
    'axes.linewidth'   : 0.7,
    'xtick.major.width': 0.7,
    'ytick.major.width': 0.7,
    'xtick.minor.width': 0.4,
    'ytick.minor.width': 0.4,
    'xtick.direction'  : 'out',
    'ytick.direction'  : 'out',
    'figure.dpi'       : 150,
    'savefig.dpi'      : 300,
})
print('Libraries loaded.')

In [ ]:
# ── Cell 2: Paths & constants ─────────────────────────────────────────────────
_DIR = os.path.dirname(os.path.abspath('__file__'))



SPATIAL_DIR = os.path.join(_DIR, 'data', 'spatial')
OUTPUT_DIR  = os.path.join(_DIR, 'output', 'figures')

# ── SSI colour palette (consistent with single-event figure) ──────────────────
THR, THR_SEV, THR_EXT = -1.28, -1.65, -2.00
C_MOD  = '#FDD692'
C_SEV  = '#F47B20'
C_EXT  = '#BE1E2D'
C_LINE = '#1a5276'
C_ENV  = 'black'
C_CTX  = '#ebebeb'
C_HEAD = '#2c4a5a'

# ── Map colour palette ────────────────────────────────────────────────────────
C_ORIGIN = '#d73027'
C_IMPL   = 'yellow'
C_NI     = '#aaaaaa'
C_OTHER  = '#555555'

# ── Layout constants ──────────────────────────────────────────────────────────
CONTEXT_DAYS = 20
STRIP_H      = 0.44   # figure-inches per SSI strip
Y_MIN, Y_MAX = -3.8, 0.3

print('Constants set.')

In [ ]:
# ── Cell 3: Load all data (run once) ─────────────────────────────────────────
print('Loading tabular data...')
ssi_all   = pd.read_csv(os.path.join(_DIR, 'data', 'SSI_daily.csv'), parse_dates=['date'])
ssi_all   = ssi_all.sort_values(['station_id', 'date'])

chains_df = pd.read_csv(os.path.join(_DIR, 'data', 'Chains_final.csv'),
                        parse_dates=['chain_start','chain_end','origin_start','origin_end'])
prop_df   = pd.read_csv(os.path.join(_DIR, 'data', 'Propagation_raw.csv'),
                        parse_dates=['origin_start','origin_end','upstream_start','upstream_end'])
prop_df['chain_id'] = (prop_df['origin_station'].astype(str) + '_' +
                       prop_df['origin_event_id'].astype(str))

conn_raw  = pd.read_csv(os.path.join(_DIR, 'data', 'upstream_connectivity.csv'),
                        sep=';', header=0,
                        names=['station_id','upstream_chain'], encoding='latin-1')
conn_raw['station_id'] = conn_raw['station_id'].astype(int)
def _parse_chain(x):
    if pd.isna(x) or str(x).strip() == '': return []
    return [int(s.strip()) for s in str(x).split(',') if s.strip().lstrip('-').isdigit()]
connectivity = dict(zip(conn_raw['station_id'],
                        conn_raw['upstream_chain'].apply(_parse_chain)))

print('Loading spatial data...')
basin_wgs   = gpd.read_file(os.path.join(SPATIAL_DIR, 'Limite_Cuenca_Ebro.shp')).to_crs(4326)
streams_wgs = gpd.read_file(os.path.join(SPATIAL_DIR, 'Stream.shp')).to_crs(4326)
sta_shp     = gpd.read_file(os.path.join(SPATIAL_DIR, 'Estaciones_finalfinal.shp')).to_crs(4326)
sta_shp['station_id'] = sta_shp['station_id'].astype(int)

print(f'Chains: {len(chains_df)}  |  Prop pairs: {len(prop_df)}')
print(f'Stations in shapefile: {len(sta_shp)}')

In [ ]:
# ── Cell 4: Interactive selection ────────────────────────────────────────────
print('Available origin stations:')
origin_stations = sorted(chains_df['origin_station'].unique())
for i, st in enumerate(origin_stations, 1):
    nc = (chains_df['origin_station'] == st).sum()
    print(f'  [{i:2d}]  Station {st}   ({nc} chains)')

idx_st    = int(input('\nSelect station [number]: ')) - 1
origin_st = origin_stations[idx_st]

avail = (chains_df[chains_df['origin_station'] == origin_st]
         .sort_values('chain_start')
         .reset_index(drop=True))

print(f'\nChains at station {origin_st}:')
for i, row in avail.iterrows():
    print(f'  [{i+1:2d}]  {row["chain_id"]}  '
          f'{row["chain_start"].strftime("%d %b %Y")} - '
          f'{row["chain_end"].strftime("%d %b %Y")}  '
          f'D = {int(row["chain_duration"])} d  '
          f'N = {int(row["chain_size"])} st  '
          f'fp = {row["propagation_fraction"]:.2f}')

idx_ch    = int(input('\nSelect chain [number]: ')) - 1
chain_row = avail.iloc[idx_ch]
chain_id  = chain_row['chain_id']
print(f'\nSelected: {chain_id}')

In [ ]:
# ── Cell 5: Build per-station records for selected chain ──────────────────────

# ── SSI strips data ───────────────────────────────────────────────────────────
sub = prop_df[prop_df['chain_id'] == chain_id].copy()
sub = (sub.sort_values(['overlap_days','upstream_duration'], ascending=[False,False])
          .drop_duplicates(subset='upstream_station', keep='first'))

records = []
for _, r in sub.iterrows():
    records.append(dict(
        station_id  = int(r['upstream_station']),
        event_start = r['upstream_start'],
        event_end   = r['upstream_end'],
        duration    = int(r['upstream_duration']),
        lag_days    = float(r['lag_days']),
        is_origin   = False,
    ))
records.append(dict(
    station_id  = int(chain_row['origin_station']),
    event_start = chain_row['origin_start'],
    event_end   = chain_row['origin_end'],
    duration    = int((chain_row['origin_end'] - chain_row['origin_start']).days + 1),
    lag_days    = 0.0,
    is_origin   = True,
))

# Sort by onset ascending: earliest upstream at top, origin at bottom
sdf = pd.DataFrame(records).sort_values('event_start').reset_index(drop=True)
n   = len(sdf)

# ── Map classification ────────────────────────────────────────────────────────
ORIGIN_STATION   = int(chain_row['origin_station'])
implicated_ids   = set(sub['upstream_station'].astype(int).unique())
all_upstream_ids = set(connectivity.get(ORIGIN_STATION, []))
non_impl_ids     = all_upstream_ids - implicated_ids
all_plotted      = all_upstream_ids | {ORIGIN_STATION}
other_ids        = set(sta_shp['station_id'].astype(int)) - all_plotted

print(f'Chain {chain_id}: {n} stations  ({n-1} upstream + 1 origin)')
print(f'Map — implicated: {len(implicated_ids)}, '
      f'non-implicated: {len(non_impl_ids)}, '
      f'other: {len(other_ids)}')

In [ ]:
# ── Cell 6: Build combined figure ────────────────────────────────────────────

# ── Shared time axis for SSI strips ──────────────────────────────────────────
t_start   = sdf['event_start'].min() - timedelta(days=CONTEXT_DAYS)
t_end     = sdf['event_end'].max()   + timedelta(days=CONTEXT_DAYS)
span_days = (t_end - t_start).days

if span_days <= 180:
    major_loc = mdates.MonthLocator(interval=1)
    minor_loc = mdates.DayLocator(interval=14)
    date_fmt  = mdates.DateFormatter('%b %Y')
elif span_days <= 730:
    major_loc = mdates.MonthLocator(interval=2)
    minor_loc = mdates.MonthLocator(interval=1)
    date_fmt  = mdates.DateFormatter('%b %Y')
else:
    major_loc = mdates.MonthLocator(interval=3)
    minor_loc = mdates.MonthLocator(interval=1)
    date_fmt  = mdates.DateFormatter('%b %Y')

# ── Map extent ────────────────────────────────────────────────────────────────
bnd = basin_wgs.total_bounds
dx, dy = bnd[2]-bnd[0], bnd[3]-bnd[1]
PAD = 0.02
x_min, x_max = bnd[0]-dx*PAD, bnd[2]+dx*PAD
y_min, y_max = bnd[1]-dy*PAD, bnd[3]+dy*PAD
mean_lat     = np.mean([y_min, y_max])
geo_asp      = 1.0 / np.cos(np.radians(mean_lat))

# ── Figure sizing ─────────────────────────────────────────────────────────────
# Height driven by number of strips; width split ~55% strips / 45% map
TOP_MARGIN    = 1.0   # inches for title
BOTTOM_MARGIN = 0.6   # inches for x-axis ticks only (legend now inside map)
FIG_H = n * STRIP_H + TOP_MARGIN + BOTTOM_MARGIN
FIG_W = 18.0

fig = plt.figure(figsize=(FIG_W, FIG_H))

# Normalised margins
top_f    = 1.0 - TOP_MARGIN    / FIG_H
bottom_f = BOTTOM_MARGIN / FIG_H

# ── GridSpec: left = SSI strips (single column), right = map ─────────────────
gs_outer = gridspec.GridSpec(
    1, 2,
    width_ratios=[1.05, 1],
    wspace=0.10,
    figure=fig,
    top=top_f, bottom=bottom_f,
    left=0.07, right=0.97,
)

# SSI strip axes (single column, all n stations)
gs_strips = gridspec.GridSpecFromSubplotSpec(
    n, 1,
    subplot_spec=gs_outer[0, 0],
    hspace=0.0,
)
strip_axes = []
for i in range(n):
    ax = fig.add_subplot(gs_strips[i, 0],
                         sharex=strip_axes[0] if i > 0 else None)
    strip_axes.append(ax)
strip_axes[0].set_xlim(t_start, t_end)

# Map axis
ax_map = fig.add_subplot(gs_outer[0, 1])

# ─────────────────────────────────────────────────────────────────────────────
#  DRAW SSI STRIPS
# ─────────────────────────────────────────────────────────────────────────────
def draw_strip(ax, row, is_last):
    st_id = row['station_id']
    ssi_st = (ssi_all[ssi_all['station_id'] == st_id]
              .set_index('date')
              .reindex(pd.date_range(t_start, t_end, freq='D'))
              .reset_index()
              .rename(columns={'index': 'date'}))
    dates = ssi_st['date'].values
    ssi   = ssi_st['SSI'].values

    ax.set_ylim(Y_MIN, Y_MAX)
    ax.set_xlim(t_start, t_end)

    # Context window
    ax.axvspan(t_start,           row['event_start'], color=C_CTX, zorder=0)
    ax.axvspan(row['event_end'],  t_end,              color=C_CTX, zorder=0)

    # Threshold reference lines
    ax.axhline(THR,     color='black',   lw=0.85, ls='--', zorder=2, alpha=0.85)
    ax.axhline(THR_SEV, color='#e67e22', lw=0.5,  ls=':',  zorder=2, alpha=0.50)
    ax.axhline(THR_EXT, color='#c0392b', lw=0.5,  ls=':',  zorder=2, alpha=0.50)

    # Severity fills within event window
    ev   = (ssi_st['date'] >= row['event_start']) & (ssi_st['date'] <= row['event_end'])
    d_ev = dates[ev]
    s_ev = ssi[ev]

    ax.fill_between(d_ev, np.maximum(s_ev, THR_SEV), THR,
                    where=s_ev < THR,     color=C_MOD, alpha=0.90, zorder=1, interpolate=True)
    ax.fill_between(d_ev, np.maximum(s_ev, THR_EXT), THR_SEV,
                    where=s_ev < THR_SEV, color=C_SEV, alpha=0.90, zorder=1, interpolate=True)
    ax.fill_between(d_ev, s_ev, THR_EXT,
                    where=s_ev < THR_EXT, color=C_EXT, alpha=0.90, zorder=1, interpolate=True)

    # SSI line within event window
    ax.plot(d_ev, s_ev, color=C_LINE, lw=0.95, zorder=3)

    # Onset marker
    ax.axvline(row['event_start'], color=C_ENV, lw=1.2, alpha=0.75, zorder=4)
    ax.plot(row['event_start'], THR, 'o', color=C_ENV, ms=4.0, zorder=5,
            markeredgewidth=0.5, markeredgecolor='white')

    # Origin station border highlight
    if row['is_origin']:
        for spine in ax.spines.values():
            spine.set_edgecolor(C_ENV)
            spine.set_linewidth(1.5)

    ax.set_yticks([THR, THR_EXT])
    ax.set_yticklabels([])
    ax.tick_params(axis='y', which='both', left=False)

    # Station label on left
    lbl = f'{st_id}' + ('  *' if row['is_origin'] else '')
    ax.set_ylabel(lbl, fontsize=7.5, rotation=0, ha='right', va='center',
                  labelpad=35,
                  fontweight='bold' if row['is_origin'] else 'normal',
                  color=C_ENV if row['is_origin'] else '#1a1a1a')

    # Right annotation: lag and duration
    lag_str = 'Origin' if row['is_origin'] else f't = {int(row["lag_days"])} d'
    dur_str = f'D = {row["duration"]} d'
    ax.annotate(f'{lag_str}\n{dur_str}',
                xy=(1.005, 0.50), xycoords='axes fraction',
                fontsize=6.0, va='center', ha='left', color='#444444',
                linespacing=1.4)

    if is_last:
        ax.xaxis.set_major_locator(major_loc)
        ax.xaxis.set_major_formatter(date_fmt)
        ax.xaxis.set_minor_locator(minor_loc)
        ax.tick_params(axis='x', which='major', labelsize=7.5)
        ax.tick_params(axis='x', which='minor', bottom=True, length=3, width=0.5)
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
    else:
        ax.tick_params(axis='x', which='both', bottom=False, labelbottom=False)

    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    ax.spines['left'].set_color('#cccccc' if not row['is_origin'] else C_ENV)
    ax.spines['bottom'].set_color('#888888' if is_last else '#dddddd')

for i, ax in enumerate(strip_axes):
    draw_strip(ax, sdf.iloc[i], is_last=(i == n - 1))

# SSI y-axis label — placed dynamically to stay clear of station ID labels
fig.canvas.draw()
renderer = fig.canvas.get_renderer()
# Left edge of ylabel texts in display pixels -> figure fraction -> step further left
min_ylabel_x_px = min(ax.yaxis.label.get_window_extent(renderer).x0 for ax in strip_axes)
xpos_ssi = min_ylabel_x_px / (FIG_W * fig.dpi) - 0.012
mid_ax   = strip_axes[len(strip_axes) // 2]
bbox     = mid_ax.get_position()
yc_ssi   = (bbox.y0 + bbox.y1) / 2
fig.text(xpos_ssi, yc_ssi, 'SSI (–)', fontsize=8, ha='center', va='center',
         rotation=90, color='#333333')

# Propagation onset envelope
pts = []
for ax, (_, row) in zip(strip_axes, sdf.iterrows()):
    xy_f = fig.transFigure.inverted().transform(
        ax.transData.transform([mdates.date2num(row['event_start']), THR])
    )
    pts.append(xy_f)

xs = np.array([p[0] for p in pts])
ys = np.array([p[1] for p in pts])
if len(xs) >= 3:
    t_   = np.linspace(0, 1, len(xs))
    t_s  = np.linspace(0, 1, 400)
    k    = min(3, len(xs) - 1)
    try:
        xs_s = make_interp_spline(t_, xs, k=k)(t_s)
        ys_s = make_interp_spline(t_, ys, k=k)(t_s)
        fig.add_artist(Line2D(xs_s, ys_s, transform=fig.transFigure,
                              color=C_ENV, lw=1.0, ls='--', alpha=0.70, zorder=11))
    except Exception:
        fig.add_artist(Line2D(xs, ys, transform=fig.transFigure,
                              color=C_ENV, lw=1.0, ls='--', alpha=0.65, zorder=11))
elif len(xs) == 2:
    fig.add_artist(Line2D(xs, ys, transform=fig.transFigure,
                          color=C_ENV, lw=1.0, ls='--', alpha=0.65, zorder=11))

# Strip panel subtitle
strip_axes[0].set_title('Upstream stations  (earliest onset → origin)',
                         fontsize=8, loc='left', color='#555555', pad=4)

# ─────────────────────────────────────────────────────────────────────────────
#  DRAW MAP
# ─────────────────────────────────────────────────────────────────────────────
ax_map.set_facecolor('white')
basin_wgs.plot(ax=ax_map, facecolor='#f2f6f9', edgecolor='black',
               linewidth=1.4, zorder=1)
streams_wgs.plot(ax=ax_map, color='#4393c3', linewidth=0.45,
                 alpha=0.80, zorder=2)

def gdf_sub(id_set):
    ids = [i for i in id_set if i in sta_shp['station_id'].values]
    return sta_shp[sta_shp['station_id'].isin(ids)]

gdf_other = gdf_sub(other_ids)
if not gdf_other.empty:
    gdf_other.plot(ax=ax_map, color=C_OTHER, markersize=14,
                   marker='o', edgecolors='#222222', linewidths=0.4, zorder=3)

gdf_ni = gdf_sub(non_impl_ids)
if not gdf_ni.empty:
    gdf_ni.plot(ax=ax_map, color=C_NI, markersize=24,
                marker='o', edgecolors='#333333', linewidths=0.5, zorder=4)

gdf_impl = gdf_sub(implicated_ids)
if not gdf_impl.empty:
    gdf_impl.plot(ax=ax_map, color=C_IMPL, markersize=32,
                  marker='o', edgecolors='black', linewidths=0.6, zorder=5)

gdf_orig = gdf_sub({ORIGIN_STATION})
if not gdf_orig.empty:
    gdf_orig.plot(ax=ax_map, color='white', markersize=58,
                  marker='o', edgecolors=C_ORIGIN, linewidths=2.2, zorder=6)
    gdf_orig.plot(ax=ax_map, color=C_ORIGIN, markersize=30,
                  marker='o', edgecolors='black', linewidths=0.7, zorder=7)

# Station labels
labelled = implicated_ids | {ORIGIN_STATION}
for _, row in sta_shp[sta_shp['station_id'].isin(labelled)].iterrows():
    ax_map.annotate(str(row['station_id']),
                    xy=(row.geometry.x, row.geometry.y),
                    xytext=(4, 3), textcoords='offset points',
                    fontsize=5.2, fontfamily='serif',
                    fontweight='bold', color='#111111', zorder=9)

# Map extent & ticks
ax_map.set_xlim(x_min, x_max)
ax_map.set_ylim(y_min, y_max)
ax_map.set_aspect(geo_asp)

def _lon_fmt(v, _):
    if   v < 0: return f'{abs(v):.0f}°W'
    elif v > 0: return f'{v:.0f}°E'
    else:       return '0°'
def _lat_fmt(v, _):
    if   v > 0: return f'{v:.0f}°N'
    elif v < 0: return f'{abs(v):.0f}°S'
    else:       return '0°'

ax_map.xaxis.set_major_locator(mticker.MultipleLocator(1.0))
ax_map.yaxis.set_major_locator(mticker.MultipleLocator(1.0))
ax_map.xaxis.set_minor_locator(mticker.MultipleLocator(0.25))
ax_map.yaxis.set_minor_locator(mticker.MultipleLocator(0.25))
ax_map.xaxis.set_major_formatter(mticker.FuncFormatter(_lon_fmt))
ax_map.yaxis.set_major_formatter(mticker.FuncFormatter(_lat_fmt))
ax_map.tick_params(axis='both', labelsize=7.0, length=4)
ax_map.tick_params(axis='both', which='minor', length=2)
ax_map.grid(True, color='#555555', alpha=0.40, linewidth=0.40,
            linestyle='--', zorder=0)

# North arrow
NA_x, NA_ytip = 0.963, 0.960
NA_ybase = NA_ytip - 0.072
ax_map.annotate('',
    xy=(NA_x, NA_ytip), xytext=(NA_x, NA_ybase),
    xycoords='axes fraction', textcoords='axes fraction',
    arrowprops=dict(arrowstyle='-|>', color='black', lw=1.3, mutation_scale=12),
    zorder=15)
ax_map.text(NA_x, NA_ybase - 0.005, 'N',
            transform=ax_map.transAxes,
            ha='center', va='top',
            fontsize=10, fontweight='bold', fontfamily='serif', zorder=15)

# Scale bar
lat0      = np.radians(mean_lat)
deg_100km = 100.0 / (np.cos(lat0) * 111.32)
n_seg     = 4
seg_deg   = deg_100km / n_seg
sb_xr     = x_max - dx * 0.015
sb_xl     = sb_xr - deg_100km
sb_y      = y_min + dy * 0.040
sb_h      = dy   * 0.016
for k in range(n_seg):
    fc = 'black' if k % 2 == 0 else 'white'
    ax_map.add_patch(mpatches.Rectangle(
        (sb_xl + k * seg_deg, sb_y - sb_h / 2),
        seg_deg, sb_h, fc=fc, ec='black', lw=0.7, zorder=14))
for k, km_val in enumerate([0, 25, 50, 75, 100]):
    ax_map.text(sb_xl + k * seg_deg, sb_y + sb_h * 0.7,
                str(km_val), ha='center', va='bottom',
                fontsize=6.0, fontfamily='serif', zorder=14)
ax_map.text((sb_xl + sb_xr) / 2, sb_y - sb_h * 1.2,
            'km', ha='center', va='top',
            fontsize=6.0, fontfamily='serif', zorder=14)

# Map legend
n_impl  = len(implicated_ids)
n_total = len(all_upstream_ids)
leg_handles_map = [
    mpatches.Patch(fc=C_ORIGIN, ec='black',   lw=0.7, label='Origin station (n=1)'),
    mpatches.Patch(fc=C_IMPL,   ec='black',   lw=0.5, label=f'Upstream implicated in chain (n={n_impl})'),
    mpatches.Patch(fc=C_NI,     ec='#333333', lw=0.4, label=f'Upstream not implicated (n={len(non_impl_ids)})'),
    mpatches.Patch(fc=C_OTHER,  ec='#222222', lw=0.4, label=f'Other stations (n={len(other_ids)})'),
]
leg_map = ax_map.legend(handles=leg_handles_map,
                         title='Station categories', title_fontsize=6.5,
                         loc='center left', bbox_to_anchor=(0.0, 0.46),
                         fontsize=6.0, framealpha=0.92, edgecolor='black',
                         fancybox=False, borderpad=0.7, handlelength=1.3)

# Chain info box
info_txt = (f'Chain: {chain_id}\n'
            f'Origin drought: {chain_row["origin_start"].strftime("%Y-%m-%d")} → '
            f'{chain_row["origin_end"].strftime("%Y-%m-%d")}\n'
            f'Chain window: {chain_row["chain_start"].strftime("%Y-%m-%d")} → '
            f'{chain_row["chain_end"].strftime("%Y-%m-%d")}  ({chain_row["chain_duration"]} d)\n'
            f'Implicated upstream: {n_impl} / {n_total}  '
            f'({chain_row["propagation_fraction"]*100:.0f}%)\n'
            f'Mean lag: {chain_row["lag_mean"]:.1f} d\n'
            f'Chain severity: {chain_row["chain_severity"]:.2f} (SSI)  |  '
            f'{chain_row["chain_severity_hm3"]:.1f} hm³')
ax_map.text(0.013, 0.013, info_txt,
            transform=ax_map.transAxes,
            fontsize=5.8, fontfamily='serif', va='bottom', linespacing=1.6,
            bbox=dict(boxstyle='round,pad=0.5', fc='white',
                      ec='#888888', lw=0.6, alpha=0.93),
            zorder=13)

ax_map.set_title('Spatial extent of the chain', fontsize=8,
                 loc='left', color='#555555', pad=4)

# ─────────────────────────────────────────────────────────────────────────────
#  SSI LEGEND — upper-right corner of the map panel
# ─────────────────────────────────────────────────────────────────────────────
legend_handles_ssi = [
    mpatches.Patch(facecolor=C_MOD, edgecolor='#aaa', label='Moderate drought (SSI < -1.28)'),
    mpatches.Patch(facecolor=C_SEV, edgecolor='#aaa', label='Severe drought (SSI < -1.65)'),
    mpatches.Patch(facecolor=C_EXT, edgecolor='#aaa', label='Extreme drought (SSI < -2.00)'),
    Line2D([0], [0], color=C_LINE, lw=1.5,          label='Daily SSI'),
    Line2D([0], [0], color=C_ENV,  lw=1.0, ls='--', label='Propagation onset envelope'),
    mpatches.Patch(facecolor=C_CTX, edgecolor='#aaa', label='Context window'),
]
# ax_map.legend() replaces the station-categories legend — so we add that back as an artist
leg_ssi = ax_map.legend(handles=legend_handles_ssi,
                         title='Legend', title_fontsize=6.5,
                         loc='lower right', bbox_to_anchor=(1.0, 1.08),
                         fontsize=6.0, framealpha=0.92, edgecolor='black',
                         fancybox=False, borderpad=0.7, handlelength=1.3, ncol=1)
ax_map.add_artist(leg_map)   # restore station-categories legend

# ─────────────────────────────────────────────────────────────────────────────
#  MAIN TITLE
# ─────────────────────────────────────────────────────────────────────────────
title_line1 = (f'Propagation chain  {chain_id}  |  '
               f'Origin: Station {int(chain_row["origin_station"])}  |  '
               f'{chain_row["chain_start"].strftime("%d %b %Y")} – '
               f'{chain_row["chain_end"].strftime("%d %b %Y")}')
title_line2 = (f'D = {int(chain_row["chain_duration"])} days  |  '
               f'N = {int(chain_row["chain_size"])} stations  |  '
               f'fp = {chain_row["propagation_fraction"]:.3f}  |  '
               f'Mean lag = {chain_row["lag_mean"]:.1f} d  |  '
               f'Severity = {chain_row["chain_severity"]:.1f}  |  '
               f'Vol. severity = {chain_row["chain_severity_hm3"]:.1f} hm³')

fig.suptitle(f'{title_line1}\n{title_line2}',
             fontsize=9, fontweight='bold', color=C_HEAD,
             y=1.0 - (TOP_MARGIN * 0.18) / FIG_H,
             linespacing=1.6)

# ─────────────────────────────────────────────────────────────────────────────
#  SAVE
# ─────────────────────────────────────────────────────────────────────────────
station_dir = os.path.join(OUTPUT_DIR, str(int(chain_row['origin_station'])))
os.makedirs(station_dir, exist_ok=True)
out_stem = os.path.join(station_dir, f'combined_chain_{chain_id}')
fig.savefig(f'{out_stem}.pdf', format='pdf', dpi=300,
            bbox_inches='tight', facecolor='white')
fig.savefig(f'{out_stem}.png', format='png', dpi=300,
            bbox_inches='tight', facecolor='white')
print(f'\nSaved: {out_stem}.pdf  and  {out_stem}.png')
plt.show()

In [ ]:
# ── Cell 7: Plot ALL chains for a selected origin station ────────────────────

def build_chain_figure(chain_row_):
    """Build and save the combined figure for one chain row from chains_df."""
    chain_id_  = chain_row_['chain_id']
    ORIGIN_ST_ = int(chain_row_['origin_station'])

    # ── Build per-station records (same logic as Cell 5) ─────────────────────
    sub_ = prop_df[prop_df['chain_id'] == chain_id_].copy()
    sub_ = (sub_.sort_values(['overlap_days','upstream_duration'], ascending=[False,False])
                .drop_duplicates(subset='upstream_station', keep='first'))

    if sub_.empty:
        print(f'  [SKIP] {chain_id_} — no propagation pairs found.')
        return

    records_ = []
    for _, r in sub_.iterrows():
        records_.append(dict(
            station_id  = int(r['upstream_station']),
            event_start = r['upstream_start'],
            event_end   = r['upstream_end'],
            duration    = int(r['upstream_duration']),
            lag_days    = float(r['lag_days']),
            is_origin   = False,
        ))
    records_.append(dict(
        station_id  = ORIGIN_ST_,
        event_start = chain_row_['origin_start'],
        event_end   = chain_row_['origin_end'],
        duration    = int((chain_row_['origin_end'] - chain_row_['origin_start']).days + 1),
        lag_days    = 0.0,
        is_origin   = True,
    ))
    sdf_ = pd.DataFrame(records_).sort_values('event_start').reset_index(drop=True)
    n_   = len(sdf_)

    # Map classification
    implicated_ids_   = set(sub_['upstream_station'].astype(int).unique())
    all_upstream_ids_ = set(connectivity.get(ORIGIN_ST_, []))
    non_impl_ids_     = all_upstream_ids_ - implicated_ids_
    other_ids_        = set(sta_shp['station_id'].astype(int)) - (all_upstream_ids_ | {ORIGIN_ST_})
    n_impl_  = len(implicated_ids_)
    n_total_ = len(all_upstream_ids_)

    # ── Time axis ─────────────────────────────────────────────────────────────
    t_start_ = sdf_['event_start'].min() - timedelta(days=CONTEXT_DAYS)
    t_end_   = sdf_['event_end'].max()   + timedelta(days=CONTEXT_DAYS)
    span_    = (t_end_ - t_start_).days
    if span_ <= 180:
        major_loc_ = mdates.MonthLocator(interval=1)
        minor_loc_ = mdates.DayLocator(interval=14)
        date_fmt_  = mdates.DateFormatter('%b %Y')
    elif span_ <= 730:
        major_loc_ = mdates.MonthLocator(interval=2)
        minor_loc_ = mdates.MonthLocator(interval=1)
        date_fmt_  = mdates.DateFormatter('%b %Y')
    else:
        major_loc_ = mdates.MonthLocator(interval=3)
        minor_loc_ = mdates.MonthLocator(interval=1)
        date_fmt_  = mdates.DateFormatter('%b %Y')

    # ── Map extent ────────────────────────────────────────────────────────────
    bnd = basin_wgs.total_bounds
    dx_, dy_ = bnd[2]-bnd[0], bnd[3]-bnd[1]
    PAD_ = 0.02
    x_min_, x_max_ = bnd[0]-dx_*PAD_, bnd[2]+dx_*PAD_
    y_min_, y_max_ = bnd[1]-dy_*PAD_, bnd[3]+dy_*PAD_
    mean_lat_ = np.mean([y_min_, y_max_])
    geo_asp_  = 1.0 / np.cos(np.radians(mean_lat_))

    # ── Figure ────────────────────────────────────────────────────────────────
    TOP_MARGIN_    = 1.0
    BOTTOM_MARGIN_ = 0.6
    FIG_H_ = n_ * STRIP_H + TOP_MARGIN_ + BOTTOM_MARGIN_
    FIG_W_ = 18.0
    fig_ = plt.figure(figsize=(FIG_W_, FIG_H_))
    top_f_    = 1.0 - TOP_MARGIN_    / FIG_H_
    bottom_f_ = BOTTOM_MARGIN_ / FIG_H_

    gs_outer_ = gridspec.GridSpec(
        1, 2, width_ratios=[1.05, 1], wspace=0.10, figure=fig_,
        top=top_f_, bottom=bottom_f_, left=0.07, right=0.97,
    )
    gs_strips_ = gridspec.GridSpecFromSubplotSpec(
        n_, 1, subplot_spec=gs_outer_[0, 0], hspace=0.0)
    strip_axes_ = []
    for i in range(n_):
        ax = fig_.add_subplot(gs_strips_[i, 0], sharex=strip_axes_[0] if i > 0 else None)
        strip_axes_.append(ax)
    strip_axes_[0].set_xlim(t_start_, t_end_)
    ax_map_ = fig_.add_subplot(gs_outer_[0, 1])

    # ── Draw SSI strips ───────────────────────────────────────────────────────
    def _draw_strip(ax, row, is_last):
        st_id = row['station_id']
        ssi_st = (ssi_all[ssi_all['station_id'] == st_id]
                  .set_index('date')
                  .reindex(pd.date_range(t_start_, t_end_, freq='D'))
                  .reset_index()
                  .rename(columns={'index': 'date'}))
        dates = ssi_st['date'].values
        ssi   = ssi_st['SSI'].values
        ax.set_ylim(Y_MIN, Y_MAX)
        ax.set_xlim(t_start_, t_end_)
        ax.axvspan(t_start_,         row['event_start'], color=C_CTX, zorder=0)
        ax.axvspan(row['event_end'], t_end_,             color=C_CTX, zorder=0)
        ax.axhline(THR,     color='black',   lw=0.85, ls='--', zorder=2, alpha=0.85)
        ax.axhline(THR_SEV, color='#e67e22', lw=0.5,  ls=':',  zorder=2, alpha=0.50)
        ax.axhline(THR_EXT, color='#c0392b', lw=0.5,  ls=':',  zorder=2, alpha=0.50)
        ev   = (ssi_st['date'] >= row['event_start']) & (ssi_st['date'] <= row['event_end'])
        d_ev = dates[ev]; s_ev = ssi[ev]
        ax.fill_between(d_ev, np.maximum(s_ev, THR_SEV), THR,
                        where=s_ev < THR,     color=C_MOD, alpha=0.90, zorder=1, interpolate=True)
        ax.fill_between(d_ev, np.maximum(s_ev, THR_EXT), THR_SEV,
                        where=s_ev < THR_SEV, color=C_SEV, alpha=0.90, zorder=1, interpolate=True)
        ax.fill_between(d_ev, s_ev, THR_EXT,
                        where=s_ev < THR_EXT, color=C_EXT, alpha=0.90, zorder=1, interpolate=True)
        ax.plot(d_ev, s_ev, color=C_LINE, lw=0.95, zorder=3)
        ax.axvline(row['event_start'], color=C_ENV, lw=1.2, alpha=0.75, zorder=4)
        ax.plot(row['event_start'], THR, 'o', color=C_ENV, ms=4.0, zorder=5,
                markeredgewidth=0.5, markeredgecolor='white')
        if row['is_origin']:
            for sp in ax.spines.values():
                sp.set_edgecolor(C_ENV); sp.set_linewidth(1.5)
        ax.set_yticks([THR, THR_EXT]); ax.set_yticklabels([])
        ax.tick_params(axis='y', which='both', left=False)
        lbl = f'{st_id}' + ('  *' if row['is_origin'] else '')
        ax.set_ylabel(lbl, fontsize=7.5, rotation=0, ha='right', va='center',
                      labelpad=35,
                      fontweight='bold' if row['is_origin'] else 'normal',
                      color=C_ENV if row['is_origin'] else '#1a1a1a')
        lag_str = 'Origin' if row['is_origin'] else f't = {int(row["lag_days"])} d'
        ax.annotate(f'{lag_str}\nD = {row["duration"]} d',
                    xy=(1.005, 0.50), xycoords='axes fraction',
                    fontsize=6.0, va='center', ha='left', color='#444444', linespacing=1.4)
        if is_last:
            ax.xaxis.set_major_locator(major_loc_)
            ax.xaxis.set_major_formatter(date_fmt_)
            ax.xaxis.set_minor_locator(minor_loc_)
            ax.tick_params(axis='x', which='major', labelsize=7.5)
            ax.tick_params(axis='x', which='minor', bottom=True, length=3, width=0.5)
            plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
        else:
            ax.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
        for sp in ['top', 'right']: ax.spines[sp].set_visible(False)
        ax.spines['left'].set_color('#cccccc' if not row['is_origin'] else C_ENV)
        ax.spines['bottom'].set_color('#888888' if is_last else '#dddddd')

    for i, ax in enumerate(strip_axes_):
        _draw_strip(ax, sdf_.iloc[i], is_last=(i == n_ - 1))

    # SSI y-axis label (dynamic positioning)
    fig_.canvas.draw()
    renderer_ = fig_.canvas.get_renderer()
    min_ylabel_x_px_ = min(ax.yaxis.label.get_window_extent(renderer_).x0 for ax in strip_axes_)
    xpos_ssi_ = min_ylabel_x_px_ / (FIG_W_ * fig_.dpi) - 0.012
    mid_ax_   = strip_axes_[len(strip_axes_) // 2]
    bbox_     = mid_ax_.get_position()
    fig_.text(xpos_ssi_, (bbox_.y0 + bbox_.y1) / 2, 'SSI (\u2013)',
              fontsize=8, ha='center', va='center', rotation=90, color='#333333')

    # Propagation envelope
    pts_ = []
    for ax, (_, row) in zip(strip_axes_, sdf_.iterrows()):
        pts_.append(fig_.transFigure.inverted().transform(
            ax.transData.transform([mdates.date2num(row['event_start']), THR])))
    xs_ = np.array([p[0] for p in pts_])
    ys_ = np.array([p[1] for p in pts_])
    if len(xs_) >= 3:
        t__ = np.linspace(0, 1, len(xs_))
        t_s_ = np.linspace(0, 1, 400)
        k_ = min(3, len(xs_) - 1)
        try:
            fig_.add_artist(Line2D(
                make_interp_spline(t__, xs_, k=k_)(t_s_),
                make_interp_spline(t__, ys_, k=k_)(t_s_),
                transform=fig_.transFigure, color=C_ENV, lw=1.0, ls='--', alpha=0.70, zorder=11))
        except Exception:
            fig_.add_artist(Line2D(xs_, ys_, transform=fig_.transFigure,
                                   color=C_ENV, lw=1.0, ls='--', alpha=0.65, zorder=11))
    elif len(xs_) == 2:
        fig_.add_artist(Line2D(xs_, ys_, transform=fig_.transFigure,
                               color=C_ENV, lw=1.0, ls='--', alpha=0.65, zorder=11))

    strip_axes_[0].set_title('Upstream stations  (earliest onset \u2192 origin)',
                              fontsize=8, loc='left', color='#555555', pad=4)

    # ── Draw map ──────────────────────────────────────────────────────────────
    ax_map_.set_facecolor('white')
    basin_wgs.plot(ax=ax_map_, facecolor='#f2f6f9', edgecolor='black', linewidth=1.4, zorder=1)
    streams_wgs.plot(ax=ax_map_, color='#4393c3', linewidth=0.45, alpha=0.80, zorder=2)

    def _gdf_sub(id_set):
        return sta_shp[sta_shp['station_id'].isin(
            [i for i in id_set if i in sta_shp['station_id'].values])]

    g = _gdf_sub(other_ids_)
    if not g.empty:
        g.plot(ax=ax_map_, color=C_OTHER, markersize=14, marker='o',
               edgecolors='#222222', linewidths=0.4, zorder=3)
    g = _gdf_sub(non_impl_ids_)
    if not g.empty:
        g.plot(ax=ax_map_, color=C_NI, markersize=24, marker='o',
               edgecolors='#333333', linewidths=0.5, zorder=4)
    g = _gdf_sub(implicated_ids_)
    if not g.empty:
        g.plot(ax=ax_map_, color=C_IMPL, markersize=32, marker='o',
               edgecolors='black', linewidths=0.6, zorder=5)
    g = _gdf_sub({ORIGIN_ST_})
    if not g.empty:
        g.plot(ax=ax_map_, color='white',  markersize=58, marker='o',
               edgecolors=C_ORIGIN, linewidths=2.2, zorder=6)
        g.plot(ax=ax_map_, color=C_ORIGIN, markersize=30, marker='o',
               edgecolors='black', linewidths=0.7, zorder=7)

    for _, row in sta_shp[sta_shp['station_id'].isin(implicated_ids_ | {ORIGIN_ST_})].iterrows():
        ax_map_.annotate(str(row['station_id']),
                         xy=(row.geometry.x, row.geometry.y),
                         xytext=(4, 3), textcoords='offset points',
                         fontsize=5.2, fontfamily='serif', fontweight='bold',
                         color='#111111', zorder=9)

    ax_map_.set_xlim(x_min_, x_max_)
    ax_map_.set_ylim(y_min_, y_max_)
    ax_map_.set_aspect(geo_asp_)

    def _lon(v, _):
        return f'{abs(v):.0f}\u00b0W' if v < 0 else (f'{v:.0f}\u00b0E' if v > 0 else '0\u00b0')
    def _lat(v, _):
        return f'{v:.0f}\u00b0N' if v > 0 else (f'{abs(v):.0f}\u00b0S' if v < 0 else '0\u00b0')

    ax_map_.xaxis.set_major_locator(mticker.MultipleLocator(1.0))
    ax_map_.xaxis.set_minor_locator(mticker.MultipleLocator(0.25))
    ax_map_.yaxis.set_major_locator(mticker.MultipleLocator(1.0))
    ax_map_.yaxis.set_minor_locator(mticker.MultipleLocator(0.25))
    ax_map_.xaxis.set_major_formatter(mticker.FuncFormatter(_lon))
    ax_map_.yaxis.set_major_formatter(mticker.FuncFormatter(_lat))
    ax_map_.tick_params(axis='both', labelsize=7.0, length=4)
    ax_map_.tick_params(axis='both', which='minor', length=2)
    ax_map_.grid(True, color='#555555', alpha=0.40, linewidth=0.40, linestyle='--', zorder=0)

    # North arrow
    ax_map_.annotate('', xy=(0.963, 0.960), xytext=(0.963, 0.888),
                     xycoords='axes fraction', textcoords='axes fraction',
                     arrowprops=dict(arrowstyle='-|>', color='black', lw=1.3, mutation_scale=12),
                     zorder=15)
    ax_map_.text(0.963, 0.883, 'N', transform=ax_map_.transAxes,
                 ha='center', va='top', fontsize=10, fontweight='bold', fontfamily='serif', zorder=15)

    # Scale bar
    lat0_    = np.radians(mean_lat_)
    deg100_  = 100.0 / (np.cos(lat0_) * 111.32)
    seg_deg_ = deg100_ / 4
    sb_xr_   = x_max_ - dx_ * 0.015
    sb_xl_   = sb_xr_ - deg100_
    sb_y_    = y_min_ + dy_ * 0.040
    sb_h_    = dy_ * 0.016
    for k in range(4):
        ax_map_.add_patch(mpatches.Rectangle(
            (sb_xl_ + k * seg_deg_, sb_y_ - sb_h_ / 2), seg_deg_, sb_h_,
            fc='black' if k % 2 == 0 else 'white', ec='black', lw=0.7, zorder=14))
    for k, km in enumerate([0, 25, 50, 75, 100]):
        ax_map_.text(sb_xl_ + k * seg_deg_, sb_y_ + sb_h_ * 0.7, str(km),
                     ha='center', va='bottom', fontsize=6.0, fontfamily='serif', zorder=14)
    ax_map_.text((sb_xl_ + sb_xr_) / 2, sb_y_ - sb_h_ * 1.2, 'km',
                 ha='center', va='top', fontsize=6.0, fontfamily='serif', zorder=14)

    # Station-categories legend
    leg_map_ = ax_map_.legend(handles=[
        mpatches.Patch(fc=C_ORIGIN, ec='black',   lw=0.7, label='Origin station (n=1)'),
        mpatches.Patch(fc=C_IMPL,   ec='black',   lw=0.5, label=f'Upstream implicated in chain (n={n_impl_})'),
        mpatches.Patch(fc=C_NI,     ec='#333333', lw=0.4, label=f'Upstream not implicated (n={len(non_impl_ids_)})'),
        mpatches.Patch(fc=C_OTHER,  ec='#222222', lw=0.4, label=f'Other stations (n={len(other_ids_)})'),
    ], title='Station categories', title_fontsize=6.5,
       loc='center left', bbox_to_anchor=(0.0, 0.46),
       fontsize=6.0, framealpha=0.92, edgecolor='black',
       fancybox=False, borderpad=0.7, handlelength=1.3)

    # Chain info box
    info_txt_ = (
        f'Chain: {chain_id_}\n'
        f'Origin drought: {chain_row_["origin_start"].strftime("%Y-%m-%d")} \u2192 '
        f'{chain_row_["origin_end"].strftime("%Y-%m-%d")}\n'
        f'Chain window: {chain_row_["chain_start"].strftime("%Y-%m-%d")} \u2192 '
        f'{chain_row_["chain_end"].strftime("%Y-%m-%d")}  ({chain_row_["chain_duration"]} d)\n'
        f'Implicated upstream: {n_impl_} / {n_total_}  '
        f'({chain_row_["propagation_fraction"]*100:.0f}%)\n'
        f'Mean lag: {chain_row_["lag_mean"]:.1f} d\n'
        f'Chain severity: {chain_row_["chain_severity"]:.2f} (SSI)  |  '
        f'{chain_row_["chain_severity_hm3"]:.1f} hm\u00b3'
    )
    ax_map_.text(0.013, 0.013, info_txt_,
                 transform=ax_map_.transAxes,
                 fontsize=5.8, fontfamily='serif', va='bottom', linespacing=1.6,
                 bbox=dict(boxstyle='round,pad=0.5', fc='white', ec='#888888', lw=0.6, alpha=0.93),
                 zorder=13)

    ax_map_.set_title('Spatial extent of the chain', fontsize=8,
                      loc='left', color='#555555', pad=4)

    # SSI legend (upper-right, outside map frame)
    leg_ssi_ = ax_map_.legend(handles=[
        mpatches.Patch(facecolor=C_MOD, edgecolor='#aaa', label='Moderate drought (SSI < \u22121.28)'),
        mpatches.Patch(facecolor=C_SEV, edgecolor='#aaa', label='Severe drought (SSI < \u22121.65)'),
        mpatches.Patch(facecolor=C_EXT, edgecolor='#aaa', label='Extreme drought (SSI < \u22122.00)'),
        Line2D([0], [0], color=C_LINE, lw=1.5, label='Daily SSI'),
        Line2D([0], [0], color=C_ENV,  lw=1.0, ls='--', label='Propagation onset envelope'),
        mpatches.Patch(facecolor=C_CTX, edgecolor='#aaa', label='Context window'),
    ], title='Legend', title_fontsize=6.5,
       loc='lower right', bbox_to_anchor=(1.0, 1.08), bbox_transform=ax_map_.transAxes,
       fontsize=6.0, framealpha=0.92, edgecolor='black',
       fancybox=False, borderpad=0.7, handlelength=1.3, ncol=1)
    ax_map_.add_artist(leg_map_)   # restore station-categories legend

    # Main title
    fig_.suptitle(
        f'Propagation chain  {chain_id_}  |  Origin: Station {ORIGIN_ST_}  |  '
        f'{chain_row_["chain_start"].strftime("%d %b %Y")} \u2013 '
        f'{chain_row_["chain_end"].strftime("%d %b %Y")}\n'
        f'D = {int(chain_row_["chain_duration"])} days  |  '
        f'N = {int(chain_row_["chain_size"])} stations  |  '
        f'fp = {chain_row_["propagation_fraction"]:.3f}  |  '
        f'Mean lag = {chain_row_["lag_mean"]:.1f} d  |  '
        f'Severity = {chain_row_["chain_severity"]:.1f}  |  '
        f'Vol. severity = {chain_row_["chain_severity_hm3"]:.1f} hm\u00b3',
        fontsize=9, fontweight='bold', color=C_HEAD,
        y=1.0 - (TOP_MARGIN_ * 0.18) / FIG_H_,
        linespacing=1.6)

    # Save
    station_dir_ = os.path.join(OUTPUT_DIR, str(ORIGIN_ST_))
    os.makedirs(station_dir_, exist_ok=True)
    out_stem_ = os.path.join(station_dir_, f'combined_chain_{chain_id_}')
    fig_.savefig(f'{out_stem_}.pdf', format='pdf', dpi=300, bbox_inches='tight', facecolor='white')
    fig_.savefig(f'{out_stem_}.png', format='png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig_)
    print(f'  Saved: combined_chain_{chain_id_}.png')


# ── Interactive: select station → plot all its chains ─────────────────────────
print('Available origin stations:')
origin_stations_all = sorted(chains_df['origin_station'].unique())
for i, st in enumerate(origin_stations_all, 1):
    nc = (chains_df['origin_station'] == st).sum()
    print(f'  [{i:2d}]  Station {st}   ({nc} chain(s))')

idx_st_all    = int(input('\nSelect station [number]: ')) - 1
origin_st_all = origin_stations_all[idx_st_all]

chains_for_st = (chains_df[chains_df['origin_station'] == origin_st_all]
                 .sort_values('chain_start')
                 .reset_index(drop=True))

print(f'\nStation {origin_st_all}: {len(chains_for_st)} chain(s) found. Building figures...\n')
for _, crow in chains_for_st.iterrows():
    print(f'  -> {crow["chain_id"]}  '
          f'{crow["chain_start"].strftime("%d %b %Y")} - {crow["chain_end"].strftime("%d %b %Y")}  '
          f'N = {int(crow["chain_size"])} stations')
    build_chain_figure(crow)

print(f'\nDone. {len(chains_for_st)} figure(s) saved to {OUTPUT_DIR}')
